In [ ]:
import legume
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

try:
    legume.Lattice
    print('You have the correct Legume')
except:
    raise Exception('Please install the correct Legume')
    #https://github.com/fancompute/legume
    #https://pypi.org/project/legume-gme/
    

# Berry phase / curvature calculations

In [ ]:
def photonic_inner_product(x1,y1,z1, x2,y2,z2, eps):
     '''
     Defines the physically meaningfull photonic inner product.
     As long as all argumetns are the same shape, this can work with any shape of np array.
     x1 y1 z1 are the components of field 1 in real space across the unit cell, and likewise for 2. EPS is material permativity.
     '''
     integrand = (x1*np.conj(x2)) + (y1*np.conj(y2)) + (z1*np.conj(z2))
     integrand *= eps
     return np.sum(integrand)




def construct_multiplet_overlap_matrix(fields_x, fields_y, fields_z, kind, kind_2):
    '''
    Helper function for the non-abelian functionality of link_u
    Defines the matrix from eq 16 in "Chern Numbers in Discretized Brillouin Zone: Efficient Method of Computing (Spin) Hall Conductances"

    Arguments are such that fields_x[i][kind] gives the x component of the eigenmode (as a full field in real space) for band i in the multiplet, asociated with the point in K space asociated with kind (k index)
    
    Returns: Square matrix, of size = multiplet size (the size of the first dimention of fields_x / y / z) 
    '''
    matrix_dimention = fields_x.shape[0]
    matrix = np.zeros([matrix_dimention, matrix_dimention], dtype=np.complex128)
    # For element i, j of the matrix ...
    for i in range(matrix_dimention):
        for j in range(matrix_dimention):
            # Take the photinic inner product between the mode in band i for the relevent point in k space, and the mode in band j for the other relevent point in k space
            matrix[i][j] = photonic_inner_product(fields_x[i][kind], fields_y[i][kind], fields_z[i][kind], fields_x[j][kind_2], fields_y[j][kind_2], fields_z[j][kind_2], eps)

    return matrix




def link_u(kx_ind, ky_ind, fields_x, fields_y, fields_z, eps, idx_map, direction = 'x', type = 'abelian'):
    '''
    Returns the U1 complex value (abs val = 1) described by eq 7 (abelian) and eq 16 (non-abelian) in "Chern Numbers in Discretized Brillouin Zone: Efficient Method of Computing (Spin) Hall Conductances"
    for a desired direction (x or y) at a point in k space defined by kx_ind ky_ind and the mapping to kind (idx_map)
    
    if type = abelian, expects fields to be ( N, (field dimentions) ) for the N total points in k space
    if type = non-abelian, expects fields to be ( m, N, (field dimentions) ) for the m bands in the multiplet

    idx_map is a square 2d numpy array that maps kx_ind and ky_ind into kind.
    This allows for the 2d BZ to be easily explored (form a loop in it), while keeping with the scheme of sequencing the eigenmodes linearly (indexed by kind).
    At k space points where kx_ind ky_ind land outside the BZ, idx_map is -1. This is important when using a hexagonal BZ. 
    '''
    # Fetches the k index for the 1st point in k space 
    kind = idx_map[kx_ind][ky_ind]
    # Fetches the k index for the next point in k space along the specified direction 
    if direction=='x':
        kind_2 = idx_map[kx_ind+1][ky_ind]
    elif direction=='y':
        kind_2 = idx_map[kx_ind][ky_ind+1]
    else: raise Exception('Must specify direction for u(1) link function')

    # if using the abelian meathod, simply perform the photonic inner product with mode kind and kind_2
    if type == 'abelian':
        quantity = photonic_inner_product(fields_x[kind], fields_y[kind], fields_z[kind], fields_x[kind_2], fields_y[kind_2], fields_z[kind_2], eps)
    
    # if using the non-abelian meathod, form the eq 16 matrix and find it's determinant
    elif type=='non-abelian':
        matrix = construct_multiplet_overlap_matrix(fields_x, fields_y, fields_z, kind, kind_2)
        quantity = np.linalg.det(matrix)

    else: raise Exception('link must be specified to use abelian or non-abelian meathod')

    # normalise output to the complex unit circle
    return quantity / np.abs(quantity)




def calculate_local_berry_phases(idx_map, fields_x, fields_y, fields_z, eps, type = 'abelian'):
    '''
    Finds the local berry phases across the BZ, as of eq 8 in "Chern Numbers in Discretized Brillouin Zone: Efficient Method of Computing (Spin) Hall Conductances"

    if type = abelian, expects fields to be ( N, (field dimentions) ) for the N total points in k space
    if type = non-abelian, expects fields to be ( m, N, (field dimentions) ) for the m bands in the multiplet

    idx_map is a square 2d numpy array that maps kx_ind and ky_ind into kind.
    This allows for the 2d BZ to be easily explored (form a loop in it), while keeping with the scheme of sequencing the eigenmodes linearly (indexed by kind).
    At k space points where kx_ind ky_ind land outside the BZ, idx_map is -1. This is important when using a hexagonal BZ.

    The result is a 1d array or floats between -pi and pi, indexed by kind
    '''
    print('Finding local berry phases')

    # instantiate an empty np array of length = (total point in k space)
    if type == 'abelian':  local_berrry_phases = np.zeros(fields_x.shape[0], dtype = float)
    elif type == 'non-abelian': local_berrry_phases = np.zeros(fields_x.shape[1], dtype = float)
    else: raise Exception('berry phase function must be specified to use abelian or non-abelian meathod')
    
    # for every point in the idx_map (indexed in kx and ky direction by kx_ind and ky_ind) ...
    for kx_ind in range(idx_map.shape[0]):
        for ky_ind in range(idx_map.shape[1]):

            # ensure we are not on the upper or right-most edge (as this means a loop will not be able to be made) (disregarding these edge eigenmodes does not meaningfully alter the results at high resolutions)
            if kx_ind >= idx_map.shape[0]-1: continue
            if ky_ind >= idx_map.shape[1]-1: continue
            # ensure that kx_ind and ky_ind lands within the BZ (this is important for non-square / non-rhombic BZ shapes, such as hexagonal)
            if idx_map[kx_ind][ky_ind]==-1: continue
            # ensure this is also the case for each point in the loop
            if idx_map[kx_ind+1][ky_ind]==-1: continue
            if idx_map[kx_ind][ky_ind+1]==-1: continue
            if idx_map[kx_ind+1][ky_ind+1]==-1: continue

            # finds the u1 link variable for each point on the loop, as of eq 8 in "Chern Numbers in Discretized Brillouin Zone: Efficient Method of Computing (Spin) Hall Conductances"
            # NOTE: we pass 'type' here ('abelian' or 'non-abelian')
            link_1 = link_u(kx_ind, ky_ind, fields_x, fields_y, fields_z, eps, idx_map, direction = 'x', type=type)
            link_2 = link_u(kx_ind+1, ky_ind, fields_x, fields_y, fields_z, eps, idx_map, direction = 'y', type=type)
            link_3 = np.conj(link_u(kx_ind, ky_ind+1, fields_x, fields_y, fields_z, eps, idx_map, direction = 'x', type=type))
            link_4 = np.conj(link_u(kx_ind, ky_ind, fields_x, fields_y, fields_z, eps, idx_map, direction = 'y', type=type))
            
            # performms the loop and finds the local berry phase (will be between -pi and pi)
            phase = np.angle(link_1*link_2*link_3*link_4)
            # saves the result to the apropriate spot in local_berry_phases
            kind = idx_map[kx_ind][ky_ind]
            local_berrry_phases[kind] = phase

    print('Found phases')
    return local_berrry_phases

# Unit cell inverse fourier transform of eigenmodes

In [ ]:
def find_modes(pwe, res, a1, a2, unit_cell='rhombic', m=0):
    '''
    Performs inverse fourier transform within the unit cell, for each eigenmode in band m, to get real-space fields.
    Also normalises these fields such that their own inner product is exactly 1 (to within floating point precision)

    Returns arrays of shape {  N, (field dimentions) for the N total points in k space }, for each compenent (as fields_x, fields_y, fields_z arrays), as fields in real space
    within the unit cell defined by the shape (unit_cell) and the a1 and a2 vectors
    Also returns eps (material permitivity in real space within the unit cell), as an array of shape (field dimentions).

    Res is the resolution of the unit cell in real space.
    pwe is the pwe object returned by legume, containing the crystal construction

    Also returns xvals and yvals, with shape (field dimentions), which map elements in the fields_x / y / z arrays with their respective coordinate in real space.
    This non-trivial implementatation is needed in order to deal with non-square / non-rhombic unit cell shapes (ie hexagonal).

    '''
    total_k_points = len(pwe.freqs[:,0])
    # Use get_rhombic_unit_cell_xy or hex_shape to find the real-space coordinates of the unit cell
    if unit_cell=='rhombic':
        xvals, yvals = get_rhombic_unit_cell_xy(a1, a2, res)
    if unit_cell=='hex':
        pts,_,_,_ = hex_shape(res, a1, a2)
        xvals, yvals = pts[:,0], pts[:,1]


    #   Perform the unit cell inverse fourier transform for the material permitivity -----
    eps_gvec = np.hstack((pwe.G1, pwe.G2, -pwe.G1, -pwe.G2)) # get g vecs
    (eps_g_unique, eps_ind_unique) = np.unique(eps_gvec, return_index=True, axis=1) # filter for unique g vecs 
    eps_ft = np.hstack(  (pwe.T1, pwe.T2, np.conj(pwe.T1), np.conj(pwe.T2))  ) # Finds the asociated coeficients (this line of code is adapted from legume source code)
    eps =  np.zeros(xvals.shape, dtype=np.complex128) # Resultant eps is real valued (within floating point precision), but this ift is a complex valued process, so they need to be stored as complex values
    for indg in eps_ind_unique:
        # forms eps field through superposition of plane waves
        eps += eps_ft[indg]*np.exp((1j*eps_gvec[0, indg]*xvals) + (1j*eps_gvec[1, indg]*yvals))


    #   Pre-computes the complex plane waves ascociated with the field g vecs. This saves a LOT of time, in comparison to re-doing this every time we ift an eigenmode -----
    field_gvec = pwe.gvec # get gvecs
    (field_g_unique, field_ind_unique) = np.unique(field_gvec, return_index=True, axis=1) # filter for unique g vecs 
    field_waves = np.zeros(np.insert(xvals.shape, 0, len(field_ind_unique)), dtype=np.complex128) # uncesesarily complex way of initialising the field_waves array with the correct shape. I need to make this more readable
    i=0
    for indg in field_ind_unique:
            # computes and stores the unit cell complex plane waves
            field_waves[i] = np.exp(  (1j*field_gvec[0, indg]*xvals) + (1j*field_gvec[1, indg]*yvals)  )
            i+=1


    #   Perform the unit cell inverse fourier transform for the e field compents -----
    fields_x, fields_y, fields_z = np.zeros(np.concatenate(([3, total_k_points],xvals.shape)), dtype=np.complex128) # initialse np arrays of zeros for each component, each of shape (total_k_points, ...field dimentions...)
    print('Doing ifft of eigenmodes:')
    for kind in range(total_k_points):
        # FOR EVERY EIGENMODE: ...

        x_ft, y_ft, z_ft = pwe.ft_field_xy('d', kind = kind, mind = m) # fetch the ft coeficients from legume
        i=0
        for indg in field_ind_unique:
            # compose the field compeonts by sums of the complex plane waves, weighted by the ft coefficients
            wave = field_waves[i]
            fields_x[kind] += x_ft[indg]*wave
            fields_y[kind] += y_ft[indg]*wave
            fields_z[kind] += z_ft[indg]*wave
            i+=1
        
        # normalises, but also randomises the global phase of every mode, FLEXING THE GAUGE INVARIENCE OF THE FUKUI MEATHOD
        norm_factor = np.real(photonic_inner_product(fields_x[kind],fields_y[kind],fields_z[kind], fields_x[kind],fields_y[kind],fields_z[kind], eps))
        fields_x[kind] /= np.sqrt(norm_factor) * np.exp(1j * np.random.random() * 2 * np.pi)
        fields_y[kind] /= np.sqrt(norm_factor) * np.exp(1j * np.random.random() * 2 * np.pi)
        fields_z[kind] /= np.sqrt(norm_factor) * np.exp(1j * np.random.random() * 2 * np.pi)

    print('Finished ifft')
    return fields_x, fields_y, fields_z, xvals, yvals, eps

# Unit cells, brillouin zones

In [ ]:
def hex_shape(res, v1, v2):
    '''
    This function is used both for creating hexagonal Brillouin zones, and also for hexagonal unit cells. In each case, v1, v2 are the reciprical lattice vectors or lattice vectors.
    Returns a pts array of shape (total points, 2),
    Returns x_grid, y_grid, np arrays of shape (res, res)

    Returns idx_map of shape (res, res)...

    ...This maps the space defined by x_grid, y_grid onto the linearised form used in the pts array (and the rest of legume).
    Parts that do not fall within the hexagon are marked in idx_map as -1. Hence, by  design, total points < res**2 

    Uses Wigner-Seitz cell
    '''
    # determine the nearest lattice point neighbours
    lat_pts = np.array([v1, v2, -1*v1, -1*v2, v1-v2, v2-v1, v1+v2, 0-v1-v2])
    # find bounding box
    min_x, min_y, max_x, max_y = np.min(lat_pts[:,0]), np.min(lat_pts[:,1]), np.max(lat_pts[:,0]), np.max(lat_pts[:,1])
    # define coordinate system
    x_grid, y_grid = np.meshgrid(np.linspace(min_x, max_x, res), np.linspace(min_y, max_y, res))
    pts = np.stack((x_grid, y_grid), axis=-1)
    
    # creates  and constructs a mask that determines wether or not the points are in or out the hexagon
    mask = np.full(x_grid.shape, True)
    d = np.linalg.norm(pts, axis=-1)
    for lat_pt in lat_pts:
        # for each neighbour lattice point, update the mask such that points which are closer to said point than (0,0) are excluded from the mask
        mask = np.logical_and(   mask, np.linalg.norm(pts-lat_pt, axis=2) >= d   )
    
    # apply mask
    pts = pts[mask]
    # create idx_map
    idx_map = np.full((res, res), -1, dtype=int)
    idx_map[mask] = np.arange(len(pts)) # leaving un-masked points as -1
    
    return pts, x_grid, y_grid, idx_map




def get_rhombic_unit_cell_xy(a1, a2, res):
    '''
    Creates res**2 total vectors that span the 2D unit cell, within the rhombus defined by a1 a2 vectors.
    Return format: X array, Y array
    '''
    r = np.linspace(0, 1, res, endpoint=False)
    r1_grid, r2_grid = np.meshgrid(r,r)
    result = (a1 * r1_grid[:,:, np.newaxis]) + (a2 * r2_grid[:,:, np.newaxis])

    return result[:,:,0], result[:,:,1]  # Flatten the (res, res, 2) 'result' np array into 1d (res**2) arrays




def square_path(K_res, centre, diamater):
    '''
    Constructs a square BZ, centred on k='centre', with specified diameter, and total points K_res ** 2
    '''
    coord_vals = np.linspace(-1*diamater/2, diamater/2, K_res)
    kx_grid, ky_grid = np.meshgrid(coord_vals + centre[0], coord_vals + centre[1])
    kx_vals, ky_vals = kx_grid.ravel(), ky_grid.ravel() 
    pts = np.stack([kx_vals, ky_vals], axis=-1)
    idx_map = np.arange(len(pts)).reshape([K_res, K_res]) # Constructs mapping of kx_ind, ky_ind to kind. This is to ensure compatablility with functions which generalise to no-square / non-rhombic BZ shapes
    return pts, kx_grid, ky_grid, idx_map




def rhombic_bz(res, b1, b2):
    '''
    Constructs a rhombic BZ. Essentially a grid in b1 b2 space from 0 to 1.
    Returns a pts array of shape (res**2, 2),
    Returns x_grid, y_grid, np arrays of shape (res, res)

    Returns idx_map of shape (res, res)...

    ...This maps the space defined by x_grid, y_grid onto the linearised form used in the pts array (and the rest of legume).
    '''
    S = np.linspace(0, 1, res, endpoint=False)
    S1Grid, S2Grid = np.meshgrid(S, S, indexing='ij')
    b1Grid = S1Grid[:,:, np.newaxis] * b1
    b2Grid = S2Grid[:,:, np.newaxis] * b2
    k_pts_grid = b1Grid + b2Grid # square array of vecors, of shape (res, res, 2)

    x_grid = k_pts_grid[:,:,0]
    y_grid = k_pts_grid[:,:,1]
    pts = np.reshape(k_pts_grid, (res**2, 2))
    idx_map = np.reshape(np.arange(res**2), (res, res))
    return pts, x_grid, y_grid, idx_map

# Crystal structure

In [ ]:
def build_valley_crystal(a1, a2, eps_rod, eps_air, d_b, d_a, pos_a, pos_b):
    '''
    Constructs the valley hall crystal as described in 'Valley-contrasting physics in all-dielectric photonic crystals: Orbital angular momentum and topological propagation'.

    Also finds the high symetry points m3, m6, k and k prime
    '''
    lattice = legume.Lattice(a1, a2)
    b1, b2 = lattice.b1, lattice.b2
    layer = legume.ShapesLayer(lattice, eps_b=eps_air)
    rod_a = legume.Circle(eps=eps_rod, x_cent=pos_a[0], y_cent=pos_a[1], r=d_a/2)
    rod_b = legume.Circle(eps=eps_rod, x_cent=pos_b[0], y_cent=pos_b[1], r=d_b/2)
    layer.add_shape([rod_a, rod_b])

    G, m3, m6, k_p, k = np.zeros(2), (b1+b2)/2, (b1+b2)/(-2), (b1+b2+b2)/(-3), (b1+b2+b2)/3
    pwe = legume.PlaneWaveExp(layer, gmax=6)
    return pwe, layer, G, m3, m6, k_p, k




def build_hex_crystal(a_R_ratio=3.135, eps_rod = 11.7, eps_air = 1.0):
    '''
    Constructs the spin hall crystal in 'Scheme for Achieving a Topological Photonic Crystal by Using Dielectric Material'
    '''
    a=1
    lattice = legume.Lattice([a/2, a*(np.sqrt(3)/2)],[a,0])
    R = 1 / a_R_ratio
    d = 2*R / 3
    
    # Add 6 dialetric circles in a regular hexagon configuration
    layer = legume.ShapesLayer(lattice, eps_b=eps_air)
    for n in range(6):
        circle = legume.Circle(eps=eps_rod, x_cent=R*np.cos(n*np.pi/3), y_cent=R*np.sin(n*np.pi/3), r=d/2)
        layer.add_shape(circle)
    pwe = legume.PlaneWaveExp(layer, gmax=7)
    return pwe, layer, lattice.a1, lattice.a2

# Plotting

In [ ]:
def valley_paper_energy_plot(layer, G, m3, m6, k_p, k, total_levels=2):
    '''
    Plots 2 graphs:
    A plot of dispersion relations (photonic energy bands) for the valley hall crystal (fig 1 b)
    A plot of the complex phase of the z component of the eigenmode at both the k point and k prime point in the BZ (this behavour is repressed if total_levels!=2)
    
    Note: G, m3, m6, k_p and k are the respecitive high-symetry-point vectors in g-space
    '''
    pwe = legume.PlaneWaveExp(layer, gmax=6)
    path_arr = np.array([m3, G, k_p, m6, G, k, m3])
    seperations = (np.sqrt(np.sum( (path_arr - np.roll(path_arr, -1, axis=0))**2, axis=1))[:-1]*10).astype(int) # Calculates how many samples to take between each high symetry point. Calculated using the euclidian distance (done in a single line using a cool trick using np.roll)
    
    # runs simulation
    path = layer.lattice.bz_path(pts=path_arr, ns=seperations)
    path['labels'] = ['M', 'Γ', "K'", 'M', 'Γ', 'K', 'M']
    pwe.run(kpoints=path['kpoints'], pol='tm')

    # actualy plots frequencies
    fig, ax = plt.subplots()
    for i in range(total_levels):
        plt.plot(pwe.freqs[:,i])
    # formatting
    ax.set_xlim([0, pwe.freqs.shape[0]-1])
    ax.set_ylabel("$\omega a/2\pi c$")
    ax.set_title(f'Photonic tm bands for crystal')
    plt.xticks(path['indexes'], path['labels'])
    ax.xaxis.grid('True')

    # New plot: Complex phase of k and k prime eigenmodes
    if total_levels==2:

        # do ift to find fields in real space in the unit cell
        fields_x, fields_y, fields_z, xgrid, ygrid, eps = find_modes(pwe, 32*3, layer.lattice.a1, layer.lattice.a2, unit_cell='hex', m=0)

        # k eigenmode part of the plot: -------------------------
        kind = np.argmin(np.linalg.norm(path['kpoints'].T - k, axis=1))

        ax.set_ylim(top=0.31, bottom=0.14) # Sets y lim of the previous plot, to mimic the look of what was put in the paper
        fig, ax  = plt.subplots()
        cbar = ax.tripcolor(xgrid.ravel(), ygrid.ravel(), np.angle(fields_z[kind]).ravel(), cmap = 'hsv', vmax = np.pi+0.1, vmin = -0.1-np.pi, shading='gouraud')
        fig.colorbar(cbar)
        ax.set_title('Arg(Ez) of mode k, showing l=1 OAM')

        # k prime eigenmode part of the plot: -------------------------
        kind = np.argmin(np.linalg.norm(path['kpoints'].T - k_p, axis=1))

        fig, ax  = plt.subplots()
        cbar = ax.tripcolor(xgrid.ravel(), ygrid.ravel(), np.angle(fields_z[kind]).ravel(), cmap = 'hsv', vmax = np.pi+0.1, vmin = -0.1-np.pi, shading='gouraud')
        fig.colorbar(cbar)
        ax.set_title("Arg(Ez) of mode k', showing l=-1 OAM")


    return fig




def hex_paper_plot(layer, a_R_ratio, total_levels = 5):
    '''
    A plot of dispersion relations (photonic energy bands) for the spin hall crystal (fig 3) from 'Scheme for Achieving a Topological Photonic Crystal by Using Dielectric Material'
    '''
    
    # construct path
    b1, b2 = layer.lattice.b1, layer.lattice.b2
    k = ((2*b1)+b2)/3
    g = np.zeros(2)
    m = b1/2
    path = layer.lattice.bz_path(pts=[k,g,m], ns=[20])

    # run simulation
    pwe = legume.PlaneWaveExp(layer, gmax=12)
    pwe.run(kpoints=path['kpoints'], pol='tm')

    # Creates plot, like in the paper the material is based from
    fig, ax = plt.subplots()
    for i in range(total_levels):
        plt.plot(pwe.freqs[:,i])
    #  formatting
    ax.set_xlim([0, pwe.freqs.shape[0]-1])
    ax.set_ylabel("$\omega a/2\pi c$")
    ax.set_title(f'Dispersion relations for the photonic crystal\nwhen a/R = {a_R_ratio}')
    plt.xticks(path['indexes'], path['labels'])
    ax.xaxis.grid('True')
    return fig


# Valley hall simulations
### Run the next 2 cells to get results:

In [ ]:
bz = 'rhombic' # BZ shape

# set to "k" for a square BZ centred on k
# set to "k'" for a square BZ centred on k prime
# set to "hex" for a hexagonal BZ (Wigner-Seitz)
# set to "rhombic" for a rhombic BZ bounded by reciprical lattice vectors

K_res = 64 # Resolution in momentum space

#  recomended value for non-hexagonal BZ: 64
#  recomended value for a hexagonal BZ: 128.  This is because the actual amount of k points is about half of what you'd expect, because it is constructed by masking out points (see hex_shape function for more details)

res = 32 # resolution in real space (recomended value, 32)

m = 0 # Simulated band (0 for lower, 1 for upper)


In [ ]:
# Constructs the valley hall crystal, according to "Valley-contrasting physics in all-dielectric photonic crystals: Orbital angular momentum and topological propagation"
a1 = np.array([1,0])
a2 = np.array([1, np.sqrt(3)]) /2
eps_rod, eps_air = 11.7, 1.0
d_a, d_b = 0.5, 0.38
pos_a = np.array([0,0])
pos_b = (a1+a2)/3

pwe, layer, G, m3, m6, k_p, k =  build_valley_crystal(a1, a2, eps_rod, eps_air, d_a, d_b, pos_a, pos_b) # (also returns the relevent high symetry points in k space)


# Constructs BZ according to desired shape. Also calculates the dA element in k space
b1, b2 = layer.lattice.b1, layer.lattice.b2
if bz=='k':
    pts, kx_grid, ky_grid, idx_map = square_path(K_res, k, 0.52*2*np.pi)
    dA = ((np.max(kx_grid) - np.min(kx_grid))) * ((np.max(ky_grid) - np.min(ky_grid))) / (K_res**2)

if bz=="k'":
    pts, kx_grid, ky_grid, idx_map = square_path(K_res, k_p, 0.52*2*np.pi)
    dA = ((np.max(kx_grid) - np.min(kx_grid))) * ((np.max(ky_grid) - np.min(ky_grid))) / (K_res**2)

if bz=="hex":
    pts, kx_grid, ky_grid, idx_map = hex_shape(K_res*3, layer.lattice.b1, layer.lattice.b2)
    dA = ((np.max(kx_grid) - np.min(kx_grid))) * ((np.max(ky_grid) - np.min(ky_grid))) / (K_res**2)

if bz=='rhombic':
    pts, kx_grid, ky_grid, idx_map = rhombic_bz(K_res, layer.lattice.b1, layer.lattice.b2)
    dA = ((b1[0]*b2[1])-(b2[0]*b1[1]) ) / (K_res**2)

# Run legume similation
path = layer.lattice.bz_path(pts=pts, ns=[1])
print('Running pwe')
pwe.run(kpoints=path['kpoints'], pol='tm')
print('Finished pwe')

# Perform unit cell inverse fourier transform of eigenmodes
fields_x, fields_y, fields_z, xgrid, ygrid, eps = find_modes(pwe, res, a1, a2, unit_cell='rhombic', m=m)

# Calculate local berry phases
local_berry_phases = calculate_local_berry_phases(idx_map, fields_x, fields_y, fields_z, eps, type = 'abelian')

# Calcualte valley chern number and band chern number integration. The use of np.abs gives the effect of constructively adding the valleys and peaks, instead of them cancling out, quantatively giving the same result as a proper double-half integration.
valley_chern_number =round( 0-np.sum(np.abs(local_berry_phases))/(np.pi)  , 2)
band_chern_number =round( np.sum(local_berry_phases)/(np.pi)  , 3)

# Produce plots, reproducing Fig 1 b and Fig 1 c in "Valley-contrasting physics in all-dielectric photonic crystals: Orbital angular momentum and topological propagation"
valley_paper_energy_plot(layer, G, m3, m6, k_p, k, total_levels=2)

# Plot berry curvature
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
ax.plot_trisurf(pts[:,0], pts[:,1], local_berry_phases/dA, cmap=cm.coolwarm, antialiased=False) # (NOTE: berry phase is divided by dA here to get berry curvature) 
# Formating:
ax.set_title(f'Berry curvature: band {m}\nBand chern number: {band_chern_number}')
if bz == 'rhombic': ax.set_title(f'Berry curvature: band {m}\nValley chern number: {valley_chern_number}\nBand chern number: {band_chern_number}')
if bz in ["k","k'"]: ax.set_title(f'Berry curvature: band {m}\n Topological index: {band_chern_number}\nCentred around {bz}')
if bz  in ['rhombic', 'hex']:
    ax.set_xlabel('kx (1/a)')
    ax.set_ylabel('ky (1/a)')
if bz == 'k':
    ax.set_xlabel('kx - Kx (1/a)')
    ax.set_ylabel('ky - Kx (1/a)')
if bz == "k'":
    ax.set_xlabel("kx - K'x (1/a)")
    ax.set_ylabel("ky - K'x (1/a)")


# Plot of eigenmode at k, with both legume and my custom ift mode overlaid. Plotting abs of z component
kind = np.argmin(local_berry_phases)
ax = legume.viz.field(pwe, 'd', kind=kind, mind=m, z=0, periodic=True, component='z', val='abs', cbar=True, eps = True).axes[0] # Fetches the legume plot ax so it can be re-used
ax.tripcolor(xgrid.ravel(), ygrid.ravel(), np.abs(fields_z[kind]).ravel(), cmap = 'magma', vmax = np.abs(fields_z[kind]).max(), vmin = 0, shading='flat') # Overlay plot with my unit cell ift
ax.set_title(f'k mode: abs(Ez), band {m}\noverlaid with unit cell')

# Plot of eigenmode at k prime, with both legume and my custom ift mode overlaid. Plotting abs of z component
kind = np.argmax(local_berry_phases)
ax = legume.viz.field(pwe, 'd', kind=kind, mind=m, z=0, periodic=True, component='z', val='abs', cbar=True, eps = True).axes[0] # Fetches the legume plot ax so it can be re-used
ax.tripcolor(xgrid.ravel(), ygrid.ravel(), np.abs(fields_z[kind]).ravel(), cmap = 'magma', vmax = np.abs(fields_z[kind]).max(), vmin = 0, shading = 'flat') # Overlay plot with my unit cell ift
ax.set_title(f"k' mode: abs(Ez), band {m}\noverlaid with unit cell")

# redundant, depricated code
'''
# Plot of eigenmode at k, with both legume and my custom ift mode overlaid. Plotting imaginary component of z component
kind = np.argmin(local_berry_phases)
ax = legume.viz.field(pwe, 'd', kind=kind, mind=m, z=0, periodic=True, component='z', val='im', cbar=True, eps = True).axes[0] # Fetches the legume plot ax so it can be re-used
ax.tripcolor(xgrid.ravel(), ygrid.ravel(), np.imag(fields_z[kind]).ravel(), cmap = 'RdBu', vmax = np.abs(fields_z[kind]).max(), vmin = -1*(np.abs(fields_z[kind]).max()), shading='flat') # Overlay plot with my unit cell ift
ax.set_title(f'k mode: Imag(Ez), band {m}\noverlaid with unit cell')

# Plot of eigenmode at k prime, with both legume and my custom ift mode overlaid. Plotting imaginary component of z component
kind = np.argmax(local_berry_phases)
ax = legume.viz.field(pwe, 'd', kind=kind, mind=m, z=0, periodic=True, component='z', val='im', cbar=True, eps = True).axes[0] # Fetches the legume plot ax so it can be re-used
ax.tripcolor(xgrid.ravel(), ygrid.ravel(), np.imag(fields_z[kind]).ravel(), cmap = 'RdBu', vmax = np.abs(fields_z[kind]).max(), vmin = -1*(np.abs(fields_z[kind]).max()), shading='flat') # Overlay plot with my unit cell ift
ax.set_title(f"k' mode: Imag(Ez), band {m}\noverlaid with unit cell")
'''


plt.show()

# Spin hall simulations (null result)
### Run the next 2 cells to get results:

In [ ]:
a_R_ratio = 2.9 # 2.9 is considered topological,  3 is the traansition point, 3.125 is non-topological

K_res = 64 # Resolution in momentum space

bands_in_multiplet = [1,2] # bands in the abelia multiplet. 1st band is 0. [1,2] is the multiplet of interest.

In [ ]:

# Construct crystal
pwe, layer, a1, a2 = build_hex_crystal(a_R_ratio, eps_rod = 11.7, eps_air = 1.0)

# PLots
legume.viz.eps(layer, cbar=True)
hex_paper_plot(layer, a_R_ratio, total_levels = 5)

# Create hexagonal BZ
pts, kx_grid, ky_grid, idx_map = hex_shape(K_res, layer.lattice.b1, layer.lattice.b2)
path = layer.lattice.bz_path(pts=pts, ns=[1])

# Run legume PWE
print('Running pwe')
pwe.run(kpoints=path['kpoints'], pol='tm')
print('Finished pwe')

# Create field multiplet
fields_x, fields_y, fields_z = [],[],[]
for m in bands_in_multiplet:
    band_fields_x, band_fields_y, band_fields_z, xvals, yvals, eps = find_modes(pwe, res, a1, a2, unit_cell='hex', m=m)
    fields_x.append(band_fields_x)
    fields_y.append(band_fields_y)
    fields_z.append(band_fields_z)
fields_x, fields_y, fields_z = np.array(fields_x), np.array(fields_y), np.array(fields_z)

# Calculate local berry phases
local_berry_phases = calculate_local_berry_phases(idx_map, fields_x, fields_y, fields_z, eps, type = 'non-abelian')

# Plot local berry phase
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
ax.plot_trisurf(pts[:,0], pts[:,1], local_berry_phases, cmap=cm.coolwarm, antialiased=False)
ax.set_title('local berry phase')
# Calculate multiplet chern number
band_chern_number =round( np.sum(local_berry_phases)/(np.pi)  , 3)
print(f'band_chern_number {band_chern_number}')

# redundant, depricated code
'''
kind = 0
mind = 1
ax = legume.viz.field(pwe, 'E', kind=kind, mind=mind, z=0, periodic=True, component='x', val='im', cbar=True, eps = True).axes[0]
ax.tripcolor(xvals, yvals, np.imag(fields_x[mind][kind]), cmap = 'RdBu', vmax = np.abs(fields_x[mind][kind]).max(), vmin = -1*(np.abs(fields_x[mind][kind]).max()))

fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
for m in [1,2]:
    ax.plot_trisurf(pts[:,0], pts[:,1], pwe.freqs[:,m], cmap=cm.coolwarm, linewidth=0, antialiased=False)
'''
plt.show()